# Cogniland Sweep Analysis

Analysis of the hyperparameter sweep over **time_penalty**, **lambda_p**, and **difficulty**.

Sweep grid:
- `time_penalty`: 0.01, 0.05, 0.10, 0.20
- `lambda_p`: 0.2, 0.5, 1.0
- `difficulty`: default (1×), diff_half (0.5×), diff_fifth (0.2× resource costs)

Total: 4 × 3 × 3 = 36 runs

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

: 

## 1. Fetch sweep runs from W&B

In [ ]:
api = wandb.Api()
runs = api.runs(
    "crusoe/cogniland",
    filters={"tags": "sweep"},
)
print(f"Found {len(runs)} sweep runs")

## 2. Build DataFrame (last-iteration metrics)

In [ ]:
WANDB_METRICS = {
    "test_det/env/success_rate":       "test_success_rate",
    "val_det/env/success_rate":        "val_success_rate",
    "test_det/env/directness_mean":    "test_directness",
    "test_det/env/exploration_mean":   "test_exploration",
    "test_det/env/risk_exposure_mean": "test_risk_exposure",
    "val_det/env/directness_mean":     "val_directness",
    "val_det/env/exploration_mean":    "val_exploration",
    "val_det/env/risk_exposure_mean":  "val_risk_exposure",
}

DIFFICULTY_MAP = {
    "default":    "default (1×)",
    "diff_half":  "half (0.5×)",
    "diff_fifth": "fifth (0.2×)",
}
DIFFICULTY_NUMERIC = {
    "default (1×)":  1.0,
    "half (0.5×)":   0.5,
    "fifth (0.2×)": 0.2,
}


def parse_difficulty(tags):
    """Extract difficulty level from W&B tags set by sweep_slurm.sh."""
    for tag in tags:
        if tag.startswith("diff_"):
            raw = tag[5:]  # strip "diff_" prefix
            return DIFFICULTY_MAP.get(raw, raw)
    return "unknown"


records = []
for run in runs:
    if run.state != "finished":
        continue
    cfg = run.config
    s = run.summary

    row = {
        "run_id":       run.id,
        "run_name":     run.name,
        "time_penalty": cfg.get("env", {}).get("time_penalty"),
        "lambda_p":     cfg.get("env", {}).get("lambda_p"),
        "difficulty":   parse_difficulty(run.tags),
    }
    for wb_key, col_name in WANDB_METRICS.items():
        row[col_name] = s.get(wb_key)

    records.append(row)

df = pd.DataFrame(records)
print(f"Collected {len(df)} finished runs")
df.head()

In [ ]:
# Quick sanity check: unique values per sweep axis
for col in ["time_penalty", "lambda_p", "difficulty"]:
    print(f"{col:>15}: {sorted(df[col].dropna().unique())}")

## 3. Sortable results table

Interactive table — click column headers to sort by test success rate, behavioral metrics, etc.

In [ ]:
TABLE_COLS = [
    "run_id", "time_penalty", "lambda_p", "difficulty",
    "test_success_rate", "val_success_rate",
    "test_directness", "test_exploration", "test_risk_exposure",
    "val_directness", "val_exploration", "val_risk_exposure",
]

df_table = (
    df[TABLE_COLS]
    .sort_values("test_success_rate", ascending=False)
    .reset_index(drop=True)
)

try:
    from itables import show
    show(
        df_table,
        paging=False,
        columnDefs=[{"className": "dt-center", "targets": "_all"}],
    )
except ImportError:
    float_cols = df_table.select_dtypes(include="number").columns
    styled = (
        df_table.style
        .format({c: "{:.4f}" for c in float_cols})
        .background_gradient(
            subset=["test_success_rate"], cmap="RdYlGn"
        )
    )
    display(styled)
    print("\n(Tip: install `itables` for interactive sorting — pip install itables)")

## 4. Effect of sweep variables on test success rate

Line plots faceted by **difficulty** to show how each reward parameter influences test performance.

### 4a. `time_penalty` vs test success rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (diff, grp) in zip(axes, df.groupby("difficulty")):
    for lp, sub in grp.groupby("lambda_p"):
        sub_sorted = sub.sort_values("time_penalty")
        ax.plot(
            sub_sorted["time_penalty"],
            sub_sorted["test_success_rate"],
            marker="o", label=f"λ_p={lp}",
        )
    ax.set_title(f"Difficulty: {diff}")
    ax.set_xlabel("time_penalty")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of time_penalty on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### 4b. `lambda_p` vs test success rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (diff, grp) in zip(axes, df.groupby("difficulty")):
    for tp, sub in grp.groupby("time_penalty"):
        sub_sorted = sub.sort_values("lambda_p")
        ax.plot(
            sub_sorted["lambda_p"],
            sub_sorted["test_success_rate"],
            marker="o", label=f"tp={tp}",
        )
    ax.set_title(f"Difficulty: {diff}")
    ax.set_xlabel("lambda_p")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of λ_p on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### 4c. `difficulty` vs test success rate

In [ ]:
DIFF_ORDER = ["default (1×)", "half (0.5×)", "fifth (0.2×)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, (lp, grp) in zip(axes, df.groupby("lambda_p")):
    for tp, sub in grp.groupby("time_penalty"):
        sub = sub.set_index("difficulty").reindex(DIFF_ORDER).reset_index()
        ax.plot(
            sub["difficulty"],
            sub["test_success_rate"],
            marker="o", label=f"tp={tp}",
        )
    ax.set_title(f"λ_p = {lp}")
    ax.set_xlabel("difficulty")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Test Success Rate")
fig.suptitle("Effect of Difficulty on Test Success Rate", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 5. Marginal effect (averaged over other axes)

Each sweep variable averaged across the other two for a clearer aggregate trend.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# time_penalty
agg = df.groupby("time_penalty")["test_success_rate"].agg(["mean", "std"]).reset_index()
axes[0].errorbar(agg["time_penalty"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[0].set_xlabel("time_penalty")
axes[0].set_ylabel("Test Success Rate")
axes[0].set_title("Marginal effect of time_penalty")

# lambda_p
agg = df.groupby("lambda_p")["test_success_rate"].agg(["mean", "std"]).reset_index()
axes[1].errorbar(agg["lambda_p"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[1].set_xlabel("lambda_p")
axes[1].set_title("Marginal effect of λ_p")

# difficulty
agg = df.groupby("difficulty")["test_success_rate"].agg(["mean", "std"]).reindex(DIFF_ORDER).reset_index()
axes[2].errorbar(range(len(agg)), agg["mean"], yerr=agg["std"], marker="o", capsize=4)
axes[2].set_xticks(range(len(agg)))
axes[2].set_xticklabels(agg["difficulty"], rotation=20)
axes[2].set_xlabel("difficulty")
axes[2].set_title("Marginal effect of difficulty")

fig.suptitle("Marginal Effects on Test Success Rate (mean ± std)", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 6. Correlation matrix: hyperparameters × behavioral metrics

Pearson correlation between the swept hyperparameters and behavioral metrics in both **validation** and **test** (deterministic policy).

Difficulty is encoded numerically as the terrain resource-cost multiplier: default=1.0, half=0.5, fifth=0.2.

In [ ]:
df_corr = df.copy()
df_corr["difficulty_num"] = df_corr["difficulty"].map(DIFFICULTY_NUMERIC)

hp_cols = ["time_penalty", "lambda_p", "difficulty_num"]
metric_cols = [
    "val_success_rate", "val_directness", "val_exploration", "val_risk_exposure",
    "test_success_rate", "test_directness", "test_exploration", "test_risk_exposure",
]

corr = df_corr[hp_cols + metric_cols].corr()
corr_subset = corr.loc[hp_cols, metric_cols]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    corr_subset,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
)
ax.set_title("Hyperparameters × Behavioral Metrics (Pearson r)", fontweight="bold")
ax.set_yticklabels(["time_penalty", "λ_p", "difficulty\n(cost multiplier)"], rotation=0)
fig.tight_layout()
plt.show()

### Full correlation matrix (all variables)

In [ ]:
all_cols = hp_cols + metric_cols
full_corr = df_corr[all_cols].corr()

labels = [
    "time_penalty", "λ_p", "difficulty",
    "val SR", "val direct.", "val explor.", "val risk",
    "test SR", "test direct.", "test explor.", "test risk",
]

mask = np.triu(np.ones_like(full_corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    full_corr,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    xticklabels=labels, yticklabels=labels,
)
ax.set_title("Full Correlation Matrix", fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Behavioral metrics by difficulty (val & test)

In [ ]:
behavioral = ["directness", "exploration", "risk_exposure"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric in zip(axes, behavioral):
    melted = pd.melt(
        df,
        id_vars=["difficulty"],
        value_vars=[f"val_{metric}", f"test_{metric}"],
        var_name="split", value_name=metric,
    )
    melted["split"] = melted["split"].str.replace(f"_{metric}", "")
    sns.boxplot(
        data=melted, x="difficulty", y=metric, hue="split",
        order=DIFF_ORDER, ax=ax, palette="Set2",
    )
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("difficulty")
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Behavioral Metrics by Difficulty (val vs test)", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()